# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thokalamanasareddy-gif/flyranktask-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import pandas as pd
import numpy as np
import pathlib
import urllib.request

# Load starter dataset (robust for local workspace OR Google Colab direct execution)
data_path = pathlib.Path('data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
    data_path = pathlib.Path('../../data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
    data_path = pathlib.Path('../data/raw/content_refresh_anonymized.csv')

if not data_path.exists():
    print("Local dataset not found. Fetching raw dataset from GitHub for Colab...")
    raw_url = "https://raw.githubusercontent.com/tejupriyakukkala-creator/flyrank-task1/main/data/raw/content_refresh_anonymized.csv"
    data_dir = pathlib.Path('data/raw')
    data_dir.mkdir(parents=True, exist_ok=True)
    data_path = data_dir / 'content_refresh_anonymized.csv'
    urllib.request.urlretrieve(raw_url, data_path)
    print(f"Successfully downloaded raw dataset to {data_path.as_posix()}")

df = pd.read_csv(data_path)

# Clean numeric columns
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'days_since_last_update', 'word_count', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Define honest label (excluded from features!)
df['is_declining_label'] = (df['trend_direction'].fillna('').str.lower() == 'down').astype(int)

# ---------------------------------------------------------
# Signal 1 Bucket Table: Days Since Last Update (Staleness Tiers)
# ---------------------------------------------------------
df['staleness_tier'] = pd.cut(df['days_since_last_update'], bins=[-1, 90, 180, 360, 9999], labels=['<90d', '90-180d', '180-360d', '360d+'])
sig1_table = df.groupby('staleness_tier', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_n=('is_declining_label', 'sum'),
    declining_rate=('is_declining_label', 'mean')
).reset_index()
sig1_table['declining_pct'] = (sig1_table['declining_rate'] * 100).round(2).astype(str) + '%'

print('=== Signal 1 Bucket Table: Content Staleness Tiers ===')
print(sig1_table.to_string(index=False))

# ---------------------------------------------------------
# Signal 2 Bucket Table: Flag-Linked Signal (Stale & Visible)
# ---------------------------------------------------------
df['stale_visible_flag'] = ((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)).astype(int)
sig2_table = df.groupby('stale_visible_flag').agg(
    n=('is_declining_label', 'count'),
    declining_n=('is_declining_label', 'sum'),
    declining_rate=('is_declining_label', 'mean')
).reset_index()
sig2_table['flag_name'] = sig2_table['stale_visible_flag'].map({0: 'Flag=0 (Other Content)', 1: 'Flag=1 (Stale >=180d & Imp >=500)'})
sig2_table['declining_pct'] = (sig2_table['declining_rate'] * 100).round(2).astype(str) + '%'
cols_sig2 = ['flag_name', 'n', 'declining_n', 'declining_rate', 'declining_pct']

print('\n=== Signal 2 Bucket Table: Flag-Linked (Stale & Visible) ===')
print(sig2_table[cols_sig2].to_string(index=False))


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import json

def percentile_rank(s):
    return s.rank(pct=True).fillna(0)

# Compute sub-scores
df['visibility_score'] = percentile_rank(np.log1p(df['impressions_90d']))
df['freshness_risk_score'] = percentile_rank(df['days_since_last_update'])
df['striking_risk_score'] = np.where((df['avg_position'] >= 4) & (df['avg_position'] <= 25), 1.0, np.where(df['avg_position'] > 25, 0.5, 0.2))
df['stale_visible_flag'] = np.where((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500), 1.0, 0.0)

# Compute transparent baseline action score
df['baseline_refresh_score'] = (
    0.35 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.20 * df['striking_risk_score']
    + 0.15 * df['stale_visible_flag']
).clip(0, 1).fillna(0)

# Reason codes mapping
def get_reasons(row):
    r = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        r.append('stale_visible_page')
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        r.append('low_ctr_visible_page')
    if 4 <= row['avg_position'] <= 20 and row['days_since_last_update'] >= 90:
        r.append('striking_decay_risk')
    if row['word_count'] > 0 and row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        r.append('thin_visible_page')
    if not r:
        r.append('general_refresh_review')
    return '|'.join(r)

def get_action(row):
    reasons = set(str(row['reason_codes']).split('|'))
    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in reasons or 'striking_decay_risk' in reasons:
        return 'refresh'
    return 'monitor'

df['reason_codes'] = df.apply(get_reasons, axis=1)
df['suggested_action_baseline'] = df.apply(get_action, axis=1)
df['baseline_rank'] = df['baseline_refresh_score'].rank(method='first', ascending=False).astype(int)

# Sort queue
df_sorted = df.sort_values('baseline_rank').reset_index(drop=True)

# Ensure output directory exists (handles Colab / local paths seamlessly)
output_dir = pathlib.Path('work/outputs')
output_dir.mkdir(parents=True, exist_ok=True)

csv_out_path = output_dir / 'baseline_action_score.csv'
json_out_path = output_dir / 'baseline_action_score_metrics.json'

output_columns = [
    'content_id',
    'client_id',
    'baseline_rank',
    'baseline_refresh_score',
    'visibility_score',
    'freshness_risk_score',
    'striking_risk_score',
    'stale_visible_flag',
    'reason_codes',
    'suggested_action_baseline',
    'is_declining_label',
    'impressions_90d',
    'clicks_90d',
    'sessions_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'word_count'
]

df_sorted[output_columns].to_csv(csv_out_path, index=False)
print(f'Wrote baseline queue to: {csv_out_path.as_posix()}')

# Calculate Precision@K vs Base Rate
base_rate = float(df['is_declining_label'].mean())
precision_metrics = {}
for k in [10, 20, 50, 100, 500]:
    p_k = float(df_sorted.head(k)['is_declining_label'].mean())
    precision_metrics[f'precision_at_{k}'] = round(p_k, 4)
    precision_metrics[f'lift_at_{k}'] = round(p_k - base_rate, 4)

metrics_payload = {
    'dataset_rows': len(df),
    'base_rate_declining': round(base_rate, 4),
    'precision_metrics': precision_metrics,
    'weights': {
        'visibility_score': 0.35,
        'freshness_risk_score': 0.30,
        'striking_risk_score': 0.20,
        'stale_visible_flag': 0.15
    }
}

with open(json_out_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)
print(f'Wrote baseline metrics JSON to: {json_out_path.as_posix()}')

# Print Precision Summary
print('\n=== Precision@K Evaluation ===')
print(f'Base Rate (Dataset Mean): {base_rate:.4f} (54.21%)')
for k in [10, 20, 50, 100, 500]:
    pk = precision_metrics[f'precision_at_{k}']
    lift = precision_metrics[f'lift_at_{k}']
    print(f'Precision@{k:3d}: {pk:.4f} ({pk*100:.1f}%) | Lift: {lift:+.4f}')


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = df_sorted.head(20).copy()

print('=== TOP-20 HAND REVIEW ===\n')
for idx, r in top20.iterrows():
    rank = int(r['baseline_rank'])
    cid = r['content_id']
    client = r['client_id']
    score = r['baseline_refresh_score']
    label = int(r['is_declining_label'])
    status = 'DECLINING (Correct Flag)' if label == 1 else 'STABLE (Weak Pick / Potential FP)'
    action = r['suggested_action_baseline']
    reasons = r['reason_codes']
    imp = int(r['impressions_90d'])
    pos = r['avg_position']
    stale = int(r['days_since_last_update'])
    words = int(r['word_count'])
    ctr = r['ctr']

    print(f"[Rank {rank:2d}] {cid} | Client: {client} | Score: {score:.4f}")
    print(f"  True Status: {status}")
    print(f"  Suggested Action: {action} | Reasons: {reasons}")
    print(f"  Metrics: Imp={imp:,} | Pos={pos:.1f} | DaysStale={stale}d | Words={words:,} | CTR={ctr:.2f}%")
    print('-' * 80)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Programmatic Feature Leakage Verification Script
forbidden_leakage_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
used_scoring_features = ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'word_count']

print('=== LEAKAGE CHECK VERIFICATION ===')
for feat in used_scoring_features:
    assert feat not in forbidden_leakage_cols, f'LEAKAGE ERROR: {feat} is in forbidden list!'
    print(f'✓ Feature verified safe: {feat}')

print('\nChecking dataframe output columns for strict feature separation...')
feature_vector_cols = [c for c in output_columns if c not in ['is_declining_label', 'baseline_rank', 'baseline_refresh_score', 'reason_codes', 'suggested_action_baseline']]
leak_detected = any(col in forbidden_leakage_cols for col in feature_vector_cols)
assert not leak_detected, 'LEAKAGE ERROR: Forbidden label column found in feature inputs!'
print('✓ Passed Leakage Audit: Zero label-derived or future-window features used in scoring!')


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.